# **Phase 1: Knowledge Base Ingestion Pipeline**
### *Project: Truth Bucket (B) Construction*
**Objective:** To build a high-fidelity dataset of 1,000+ Egyptian news articles for RAG-based analysis.
**Key Features:**
* **RSS Auto-Discovery:** Real-time link extraction from major news domains.
* **Surgical Cleaning:** Regex-based removal of sidebar junk and trending news leakages.
* **Stealth Scraping:** Anti-bot measures to bypass HTTP 403 blocks.
---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install necessary libraries for the Ingestion Pipeline
!pip install feedparser
print("✅ Libraries installed.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.1 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=80679f864b357cb1b8ce00793b7faef70687cb97d397136900ed57c6a4457bc5
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
✅ Libraries installed.


In [ ]:
import requests
import feedparser
import pandas as pd
from datetime import datetime
import os
import time
import random
import re
from tqdm import tqdm
from bs4 import BeautifulSoup
from google.colab import drive

## **Phase 1: Real-Time Discovery (RSS Ingestion)**
> *Goal: Continuously monitor 10+ Egyptian news feeds to identify the latest 'Ground Truth' articles.*
* **Logic:** Scans XML feeds from Youm7, Al-Masry, and RT.
* **Metadata:** Captures Title, Link, and Timestamp for every news entry.

In [ ]:
NEWS_SOURCES = {
    # --- 1. THE GOLD MINES (Highest Volume & Stability) ---
    "RT_Arabic_Main": "https://arabic.rt.com/rss/",
    "RT_MiddleEast": "https://arabic.rt.com/rss/middle_east/",
    "RT_Egypt": "https://arabic.rt.com/rss/egypt/",
    "RT_Business": "https://arabic.rt.com/rss/business/",
    "France24_Arabic": "https://www.france24.com/ar/rss",
    "DW_Arabic": "https://www.dw.com/ar/rss",

    # --- 2. EGYPTIAN RELIABLES (Working in your logs) ---
    "AlMasry_Latest": "https://www.almasryalyoum.com/rss/rssfeed",
    "Masry_Politics": "https://www.almasryalyoum.com/rss/rssfeed?sectionId=1",
    "Masry_Economy": "https://www.almasryalyoum.com/rss/rssfeed?sectionId=4",
    "Masry_Sports": "https://www.almasryalyoum.com/rss/rssfeed?sectionId=5",
    "BBC_Arabic": "http://feeds.bbci.co.uk/arabic/rss.xml",
    "CNN_Arabic": "https://arabic.cnn.com/api/v1/rss/world/rss.xml",

    # --- 3. THE RECOVERY SOURCES (Fixed Paths for 2026) ---
    "AlArabiya_Latest": "https://www.alarabiya.net/.mrss/ar/last-24-hours.xml",
    "Asharq_Latest": "https://asharq.com/rss/latest/",
    "AlJazeera_Main": "https://www.aljazeera.net/aljazeerarss/a7c186be-1baa-4bd4-9d80-a233303400d1/73c17b40-e513-4bd2-9a20-a358d3f5a542",
    "Youm7_Politics": "https://www.youm7.com/rss/SectionRss?SectionID=97",
    "Youm7_Breaking": "https://www.youm7.com/rss/SectionRss?SectionID=65",
    "SkyNews_Latest": "https://www.skynewsarabia.com/rss/feeds/rss_6.xml",

    # --- 4. THE VOLUME BOOSTERS (Stable Aggregators) ---
    "Dostor_News": "https://www.dostor.org/rss.aspx",
    "Veto_Gate": "https://www.vetogate.com/rss.aspx",
    "Sada_ElBalad": "https://www.elbalad.news/Rss/Rss",
    "Ahram_Latest": "https://gate.ahram.org.eg/rss/latest.aspx"
}

In [ ]:
def ingest_truth_bucket_robust(sources):
    all_data = []
    # This 'Header' makes the website think you are a real person on Chrome
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    print(f"🚀 Starting Ingestion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    for name, url in sources.items():
        try:
            print(f"📡 Fetching live updates from {name}...")
            # We use 'requests' first to bypass security, then 'feedparser'
            response = requests.get(url, headers=headers, timeout=10)
            feed = feedparser.parse(response.content)

            if not feed.entries:
                print(f"⚠️ Warning: {name} returned 0 results. Checking another link...")
                continue

            for entry in feed.entries:
                all_data.append({
                    "source": name,
                    "title": entry.title,
                    "summary": entry.get("summary", "Summary not available"),
                    "link": entry.link,
                    "timestamp": entry.get("published", datetime.now().strftime('%Y-%m-%d')),
                    "bucket": "B_Truth"
                })
            print(f"✅ Success: Found {len(feed.entries)} articles from {name}")

        except Exception as e:
            print(f"❌ Error fetching {name}: {e}")

    return pd.DataFrame(all_data)

# Run the process again
df_kb_truth = ingest_truth_bucket_robust(NEWS_SOURCES)

🚀 Starting Ingestion: 2026-04-26 23:39:27
📡 Fetching live updates from RT_Arabic_Main...
✅ Success: Found 100 articles from RT_Arabic_Main
📡 Fetching live updates from RT_MiddleEast...
✅ Success: Found 50 articles from RT_MiddleEast
📡 Fetching live updates from RT_Egypt...
⚠️ Warning: RT_Egypt returned 0 results. Checking another link...
📡 Fetching live updates from RT_Business...
✅ Success: Found 50 articles from RT_Business
📡 Fetching live updates from France24_Arabic...
✅ Success: Found 24 articles from France24_Arabic
📡 Fetching live updates from DW_Arabic...
⚠️ Warning: DW_Arabic returned 0 results. Checking another link...
📡 Fetching live updates from AlMasry_Latest...
✅ Success: Found 20 articles from AlMasry_Latest
📡 Fetching live updates from Masry_Politics...
✅ Success: Found 20 articles from Masry_Politics
📡 Fetching live updates from Masry_Economy...
✅ Success: Found 20 articles from Masry_Economy
📡 Fetching live updates from Masry_Sports...
✅ Success: Found 20 articles fro

In [ ]:
# Define your Drive path
final_path = "/content/drive/MyDrive/GP_Project/Bucket_B_Truth_CLEAN_FINAL.csv"

def save_and_update_kb(new_df, path):
    if os.path.exists(path):
        # 1. Load the existing data (the 134 articles from yesterday)
        existing_df = pd.read_csv(path)
        # 2. Combine with the new findings
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        # 3. Deduplicate based on the 'link' to ensure we only have unique news
        final_df = combined_df.drop_duplicates(subset=['link'], keep='first')
        print(f"🔄 Growth Logic Active: Total unique records in Bucket B: {len(final_df)}")
    else:
        final_df = new_df
        print(f"🆕 Initializing Bucket B: {len(final_df)} records saved.")

    # Save the updated master list back to Drive
    final_df.to_csv(path, index=False, encoding='utf-8-sig')
    return final_df

# Call the function
df_kb_truth = save_and_update_kb(df_kb_truth, final_path)

🔄 Growth Logic Active: Total unique records in Bucket B: 1384


In [ ]:
# Show the organized data to the professor
df_kb_truth[['source', 'title', 'timestamp']].head(10)

,source,title,timestamp
0,BBC_Arabic,"ترامب يؤكد وجود ""محادثات جدية"" مع إيران، وطهرا...",2026-04-18
1,BBC_Arabic,عندما يصبح الرجل أباً: كيف تعيد الأبوة تشكيل د...,"Sat, 18 Apr 2026 20:11:09 GMT"
2,BBC_Arabic,"رئيس ""أنثروبيك"" للذكاء الاصطناعي يلتقي مسؤولين...","Sat, 18 Apr 2026 19:28:11 GMT"
3,BBC_Arabic,عهد أوربان انتهى في لمح البصر، ورئيس وزراء الم...,"Sat, 18 Apr 2026 13:38:05 GMT"
4,BBC_Arabic,من فيروز ووديع الصافي إلى جوليا: حكاية الجنوب ...,"Sat, 18 Apr 2026 05:59:27 GMT"
5,BBC_Arabic,ما تعلمته عندما توقفت عن تناول السكر لستة أسابيع,"Sat, 18 Apr 2026 08:24:00 GMT"
6,BBC_Arabic,مع بدء الهدنة في لبنان، قيادي رفيع لبي بي سي: ...,"Fri, 17 Apr 2026 19:12:41 GMT"
7,BBC_Arabic,ما الذي نعرفه عن وقف إطلاق النار بين لبنان وإس...,"Fri, 17 Apr 2026 07:51:52 GMT"
8,BBC_Arabic,هل خففت إيران قيود فرض الحجاب على النساء؟,"Fri, 17 Apr 2026 12:50:08 GMT"
9,BBC_Arabic,ماذا نعرف عن الألغام البحرية وكاسحاتها؟,"Wed, 15 Apr 2026 15:14:09 GMT"


## **Step 2: Surgical Extraction & Stealth Logic**
This module contains the "brain" of the scraper.
* **`clean_text_final`**: Uses "Kill-Switch" logic to stop reading text once it hits sidebar junk (e.g., *Zamalek news* or *Trending blocks*).
* **`stealth_scraper`**: Mimics a real browser using rotating `User-Agents` and `Referers` to avoid being blocked by news site security.

In [ ]:
!pip install beautifulsoup4

In [ ]:
import random, re, time, requests
from bs4 import BeautifulSoup
from tqdm import tqdm

def clean_text_final_V4(text):
    if not isinstance(text, str) or len(text) < 10: return text

    # 1. NEW: RT-Arabic Specific Cleaner
    # RT articles always start after the 'GMT' timestamp or 'تاريخ النشر'
    if "GMT" in text:
        # Split by GMT and take everything AFTER it
        parts = text.split("GMT")
        if len(parts) > 1:
            text = parts[1]

    # 2. General Cleanup for the sidebar menu noise
    menu_noise = [
        r"Stories \d+ خبر",
        r"مشاهدة فيديوهات",
        r"مشاهدة هدنة",
        r"مشاهدة العملية العسكرية",
        r"حصار المضيق",
        r"نبض الملاعب"
    ]
    for pattern in menu_noise:
        text = re.sub(pattern, "", text)

    # 3. Existing 'Stop Phrases' (The Kill-Switch)
    stop_phrases = ["لا يفوتك", "ترند", "المعلم وعباس", "ميدو وحازم", "المصدر:", "اقرأ أيضا", "طالع أيضا", "اليوم السابع"]
    for phrase in stop_phrases:
        if phrase in text: text = text.split(phrase)[0]

    # 4. Strip HTML tags and Social Media
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'Facebook|X|Telegram|Share|انسخ الرابط|تويتر|فيسبوك', "", text, flags=re.IGNORECASE)

    # 5. Clean whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def standard_news_scraper(df, target_only_failed=False, limit=500):
    """General purpose scraper with built-in cleaning and fallback."""
    if target_only_failed:
        mask = df['full_text'].str.contains("Failed|Error|403|^$", na=True)
    else:
        mask = df['full_text'].isna() | (df['full_text'] == "")

    pending_indices = df[mask].index
    to_process = pending_indices[:limit]

    print(f"🚀 Scraping {len(to_process)} articles...")

    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
        'Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Mobile/15E148 Safari/604.1'
    ]

    for idx in tqdm(to_process):
        url = df.loc[idx, 'link']
        try:
            time.sleep(random.uniform(3, 5))
            headers = {'User-Agent': random.choice(user_agents), 'Referer': 'https://www.google.com/'}
            response = requests.get(url, headers=headers, timeout=15)

            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')

                # Adaptive search
                body = soup.find('div', {'id': ['article-body', 'story-body', 'main-content']}) or \
                       soup.find('div', {'class': ['article-body', 'story-text', 'details-content', 'article-content']}) or \
                       soup.find('article')

                if body:
                    raw_text = body.get_text(separator=' ', strip=True)
                else:
                    # Paragraph Brute Force
                    paragraphs = [p.get_text().strip() for p in soup.find_all('p') if len(p.get_text()) > 50]
                    raw_text = " ".join(paragraphs)

                if len(raw_text) > 50:
                    df.at[idx, 'full_text'] = clean_text_final_V4(raw_text)
                else:
                    df.at[idx, 'full_text'] = "Failed: Still No Container"
            else:
                df.at[idx, 'full_text'] = f"Failed: HTTP {response.status_code}"
        except Exception as e:
            df.at[idx, 'full_text'] = f"Error: {str(e)[:20]}"

    return df

# --- EXECUTION & SAVING ---
# 1. Scrape the new articles
df_kb_truth = standard_news_scraper(df_kb_truth, limit=100)

# 2. Final Purge (Remove rows that simply won't work for the meeting)
df_kb_truth = df_kb_truth[~df_kb_truth['full_text'].str.contains("Failed:|Error:|HTTP 403", na=False)]
df_kb_truth = df_kb_truth[df_kb_truth['full_text'].str.len() > 150]

# 3. Clean the Summary column too (Removes the HTML tags you mentioned)
df_kb_truth['summary'] = df_kb_truth['summary'].apply(clean_text_final_V4)

# 4. Save to the Final Project Path
final_path = "/content/drive/MyDrive/GP_Project/Bucket_B_Truth_CLEAN_FINAL.csv"
df_kb_truth.to_csv(final_path, index=False, encoding='utf-8-sig')

print(f"✅ Final KB Saved! Total Articles for Meeting: {len(df_kb_truth)}")

🚀 Scraping 100 articles...


100%|██████████| 100/100 [08:40<00:00,  5.21s/it]

✅ Final KB Saved! Total Articles for Meeting: 1016


In [ ]:
# Install Transformers and Arabic preprocessing tools
!pip install transformers farasapy pyarabic torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 4.3 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load the AraBERT model and tokenizer
model_name = "aubmindlab/bert-base-arabertv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Move to GPU if available for speed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(64000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
import numpy as np

def get_bert_embeddings(text_list, batch_size=16):
    model.eval()
    all_embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]

        # Tokenize and move to device
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Use the [CLS] token (first token) as the summary of the whole sentence
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

In [ ]:
from google.colab import drive
import pandas as pd

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Load the CSV from your specific path
# Change 'My Drive/...' to the actual folder where you saved the file
csv_path = '/content/drive/My Drive/GP_Project/Bucket_B_Truth_CLEAN_FINAL.csv'

try:
    df_kb_truth = pd.read_csv(csv_path)
    print("✅ Success! Knowledge Base loaded from Drive.")
    print(f"Total articles found: {len(df_kb_truth)}")
except Exception as e:
    print(f"❌ Error: Could not find the file. Please check the path. Error: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Success! Knowledge Base loaded from Drive.
Total articles found: 1384


In [ ]:
import pandas as pd
import re
import os
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/My Drive/GP_Project/Bucket_B_Truth_CLEAN_FINAL.csv'
df_kb_truth = pd.read_csv(file_path)
print(f'Loaded {len(df_kb_truth)} articles.')

def extract_propositions(text):
    if not isinstance(text, str): return []
    sentences = re.split(r'(?<=[.!?\u061f])\s+', text)
    return [s.strip() for s in sentences if len(s.split()) > 5]

propositions_list = []
for idx, row in df_kb_truth.iterrows():
    source_text = row['full_text'] if pd.notna(row.get('full_text')) else row.get('summary', '')
    props = extract_propositions(source_text)
    for p in props:
        propositions_list.append({
            'parent_article_id': idx,
            'parent_title':      row['title'],
            'proposition_child': p,
            'source':            row['source'],
            'link':              row['link'],
            'timestamp':         row['timestamp'],
            'bucket':            row['bucket']
        })

df_propositions = pd.DataFrame(propositions_list)
print(f'Parent articles : {df_kb_truth.shape[0]}')
print(f'Child propositions: {len(df_propositions)}')
print(df_propositions[['parent_title','source','proposition_child']].head(3))


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

# 1. Initialize AraBERT
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embeddings(text_list):
    """Encodes a list of sentences into AraBERT vectors"""
    inputs = tokenizer(text_list, padding=True, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # Use Mean Pooling to get a single vector per sentence
    return outputs.last_hidden_state.mean(dim=1).numpy()

print(f"⏳ Starting embedding for {len(df_propositions)} propositions...")

# 2. Process in batches to avoid memory errors
batch_size = 32
all_embeddings = []

for i in range(0, len(df_propositions), batch_size):
    batch_text = df_propositions['proposition_child'].iloc[i:i+batch_size].tolist()
    batch_emb = get_embeddings(batch_text)
    all_embeddings.append(batch_emb)
    if i % 160 == 0:
        print(f"📡 Progress: {i}/{len(df_propositions)} propositions encoded.")

proposition_vectors = np.vstack(all_embeddings)

# 3. Save the vectors to your Drive for tomorrow
np.save('/content/drive/My Drive/proposition_vectors.npy', proposition_vectors)
df_propositions.to_csv('/content/drive/My Drive/df_propositions_mapped.csv', index=False)

print("-" * 30)
print("✅ SUCCESS: Embeddings generated and saved to Drive.")
print("📊 Vector Shape:", proposition_vectors.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Starting embedding for 9312 propositions...
📡 Progress: 0/9312 propositions encoded.
📡 Progress: 160/9312 propositions encoded.
📡 Progress: 320/9312 propositions encoded.
📡 Progress: 480/9312 propositions encoded.
📡 Progress: 640/9312 propositions encoded.
📡 Progress: 800/9312 propositions encoded.
📡 Progress: 960/9312 propositions encoded.
📡 Progress: 1120/9312 propositions encoded.
📡 Progress: 1280/9312 propositions encoded.
📡 Progress: 1440/9312 propositions encoded.
📡 Progress: 1600/9312 propositions encoded.
📡 Progress: 1760/9312 propositions encoded.
📡 Progress: 1920/9312 propositions encoded.
📡 Progress: 2080/9312 propositions encoded.
📡 Progress: 2240/9312 propositions encoded.
📡 Progress: 2400/9312 propositions encoded.
📡 Progress: 2560/9312 propositions encoded.
📡 Progress: 2720/9312 propositions encoded.
📡 Progress: 2880/9312 propositions encoded.
📡 Progress: 3040/9312 propositions encoded.
📡 Progress: 3200/9312 propositions encoded.
📡 Progress: 3360/9312 propositions enco

In [ ]:
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

# 1. Load the data and vectors we created earlier
# (Assuming proposition_vectors.npy and df_propositions are ready)
vectors = np.load('/content/drive/My Drive/proposition_vectors.npy')
df_props = pd.read_csv('/content/drive/My Drive/GP_Project/df_propositions_mapped.csv')

# 2. Run K-Means to find the natural groupings
num_clusters = 8  # You can adjust this based on how many topics you want
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_props['cluster_id'] = kmeans.fit_predict(vectors)

print("✅ Clustering Complete.")

# 3. IDENTIFY THE "CENTROIDS" (The 'Beyond K-Means' Step)
# For each cluster, we find the sentence closest to the mathematical center
from sklearn.metrics import pairwise_distances_argmin_min

closest_indices, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, vectors)

cluster_map = []
for i, idx in enumerate(closest_indices):
    centroid_sentence = df_props.iloc[idx]['proposition_child']
    cluster_map.append({
        'Cluster_ID': i,
        'Centroid_Summary': centroid_sentence, # This is the "Title" of the cluster
        'Size': len(df_props[df_props['cluster_id'] == i])
    })

df_cluster_summary = pd.DataFrame(cluster_map)

# 4. Save this for your presentation
df_cluster_summary.to_csv('/content/drive/My Drive/Knowledge_Map_Summary.csv', index=False)

print("\n🚀 YOUR KNOWLEDGE MAP IS READY FOR 11 AM:")
print(df_cluster_summary[['Cluster_ID', 'Size', 'Centroid_Summary']])

✅ Clustering Complete.

🚀 YOUR KNOWLEDGE MAP IS READY FOR 11 AM:
   Cluster_ID  Size                                   Centroid_Summary
0           0   608  الأرصاد تحذر من حالة الطقس اليوم الخميس 16-4-2...
1           1  1568  وقال نتنياهو إنه يريد توسيع ما يسميه "المنطقة ...
2           2  1770  إسرائيل تهاجم فرنسا في الأمم المتحدة وسط جدل ع...
3           3  1042  إذ رأى أن النظام الجديد لا يقدم فائدة تُذكر لم...
4           4  1673  ورغم تحفظه الجزئي على هذا الطرح، أكد البرغوثي ...
5           5   869  ومضيق هرمز خط أحمر حتى استعادة الحقوق الولايات...
6           6  1656  وفي مستهل كلمته، رحب أبو العينين بالمشاركين، م...
7           7   126  ​مدن القناة وشمال سيناء: ​الإسماعيلية: الصغرى ...


In [ ]:
from sklearn.metrics import silhouette_score

# 1. Calculate the Silhouette Score
# (We use a sample of 5000 if the dataset is huge to save time,
# but 9312 should run fine in a few minutes)
print("🧪 Calculating Silhouette Score (Validation)...")
score = silhouette_score(vectors, df_props['cluster_id'], metric='cosine')

# 2. Calculate Inertia (Elbow Method value)
inertia = kmeans.inertia_

print("-" * 30)
print(f"📊 CLUSTERING VALIDATION RESULTS:")
print(f"✅ Silhouette Score: {score:.4f}")
print(f"✅ Cluster Inertia: {inertia:.2f}")
print("-" * 30)

if score > 0.10:
    print("Interpretation: Significant improvement over the previous article-level clustering!")
else:
    print("Interpretation: The clusters are defined, but there is some thematic overlap.")

🧪 Calculating Silhouette Score (Validation)...
------------------------------
📊 CLUSTERING VALIDATION RESULTS:
✅ Silhouette Score: 0.0510
✅ Cluster Inertia: 1487188.25
------------------------------
Interpretation: The clusters are defined, but there is some thematic overlap.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Intra-Cluster Similarity Check
print("🔍 Calculating Internal Cohesion...")
intra_similarities = []

for i in range(num_clusters):
    cluster_indices = df_props[df_props['cluster_id'] == i].index
    cluster_vecs = vectors[cluster_indices]
    centroid_vec = kmeans.cluster_centers_[i].reshape(1, -1)

    # Calculate similarity of each point to the center
    sims = cosine_similarity(cluster_vecs, centroid_vec)
    intra_similarities.append(np.mean(sims))

# 2. Parent Consistency Check (The 'Golden' Metric)
print("🔍 Checking Parent-Child Logic...")
consistency_scores = []
# Group by parent and see how many clusters their children are spread across
for parent_id, group in df_props.groupby('parent_article_id'):
    # What % of the children are in the most frequent cluster for this parent?
    top_cluster_count = group['cluster_id'].value_counts().iloc[0]
    consistency = top_cluster_count / len(group)
    consistency_scores.append(consistency)

avg_consistency = np.mean(consistency_scores)

print("\n" + "="*40)
print("🏆 NEW VALIDATION METRICS FOR DR. CHERRY")
print(f"✅ Avg. Intra-Cluster Similarity: {np.mean(intra_similarities):.4f}")
print(f"✅ Parent-Child Consistency Score: {avg_consistency:.2%}")
print("="*40)

# Create a final table for the slide
df_cluster_summary['Cohesion_Score'] = intra_similarities
print(df_cluster_summary[['Cluster_ID', 'Size', 'Cohesion_Score', 'Centroid_Summary']])

🔍 Calculating Internal Cohesion...
🔍 Checking Parent-Child Logic...

🏆 NEW VALIDATION METRICS FOR DR. CHERRY
✅ Avg. Intra-Cluster Similarity: 0.8035
✅ Parent-Child Consistency Score: 52.92%
   Cluster_ID  Size  Cohesion_Score  \
0           0   608        0.830043   
1           1  1568        0.804062   
2           2  1770        0.671977   
3           3  1042        0.791384   
4           4  1673        0.819450   
5           5   869        0.866429   
6           6  1656        0.738967   
7           7   126        0.905304   

                                    Centroid_Summary  
0  الأرصاد تحذر من حالة الطقس اليوم الخميس 16-4-2...  
1  وقال نتنياهو إنه يريد توسيع ما يسميه "المنطقة ...  
2  إسرائيل تهاجم فرنسا في الأمم المتحدة وسط جدل ع...  
3  إذ رأى أن النظام الجديد لا يقدم فائدة تُذكر لم...  
4  ورغم تحفظه الجزئي على هذا الطرح، أكد البرغوثي ...  
5  ومضيق هرمز خط أحمر حتى استعادة الحقوق الولايات...  
6  وفي مستهل كلمته، رحب أبو العينين بالمشاركين، م...  
7  ​مدن القناة وشم

In [ ]:
# Change cluster_to_check to see different topics
cluster_to_check = 0

print(f"🧐 DEEP CHECK: TOPIC SAMPLES FOR CLUSTER {cluster_to_check}")
samples = df_props[df_props['cluster_id'] == cluster_to_check]['proposition_child'].sample(5).values

for i, s in enumerate(samples):
    print(f"[{i+1}] {s[:150]}...") # Printing first 150 chars

🧐 DEEP CHECK: TOPIC SAMPLES FOR CLUSTER 0
[1] Play تحميل الفيديو الطاقة طهران مضيق هرمز ناقلات النفط...
[2] أخبار متعلقة «خلي بالك أوعى تأكلها».....
[3] المصدر : ذا ماركر أخبار مصر أخبار مصر اليوم القاهرة غوغل - Google Save post VK.com التعليقات شارك بالتعليقات هام لحظة بلحظة.....
[4] تمثال بوموس للخصوبة القرن العشرين قرية بوموس قبرص تمثال صليب منحوت تمثال بوموس الخصوبة العصر النحاسي تابعوا آخر أخبار...
[5] حيث سجل عيار 21 سعر 7040جنيها وننشر اخر تطورات سعر الذهب اليوم السبت في مصر....


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. استخراج الكلمات المفتاحية لكل Cluster (لإثبات الدقة للدكتورة)
def get_cluster_keywords(cluster_id, top_n=10):
    cluster_text = df_props[df_props['cluster_id'] == cluster_id]['proposition_child'].tolist()
    vectorizer = TfidfVectorizer(max_features=100)
    tfidf_matrix = vectorizer.fit_transform(cluster_text)
    feature_names = vectorizer.get_feature_names_out()
    # الحصول على الكلمات الأكثر أهمية
    sums = tfidf_matrix.sum(axis=0)
    data = []
    for col, capability in enumerate(feature_names):
        data.append((capability, sums[0, col]))
    ranking = sorted(data, key=lambda x: x[1], reverse=True)
    return [word for word, score in ranking[:top_n]]

# تجربة على Cluster 0 و Cluster 1
print("🏷️ تحليل المحتوى الفعلي للمجموعات:")
for i in range(2):
    keywords = get_cluster_keywords(i)
    print(f"Cluster {i} Keywords: {', '.join(keywords)}")

🏷️ تحليل المحتوى الفعلي للمجموعات:
Cluster 0 Keywords: أخبار, في, من, آخر, تابعوا, شارك, 2026, طباعة, الرئيسية, اليوم
Cluster 1 Keywords: في, من, على, أن, إلى, إيران, مع, عن, ترامب, المتحدة


In [ ]:
noise_keywords = [
    'فيديو', 'تحميل', 'شارك', 'التعليقات', 'جهازك', 'تطبيقات',
    'الرئيسية', 'سجل', 'موقع', 'تابعوا', 'طباعة', 'VK', 'Google'
]

mask = df_propositions['proposition_child'].str.len() > 25
for word in noise_keywords:
    mask = mask & (~df_propositions['proposition_child'].str.contains(word, na=False))

df_clean = df_propositions[mask].reset_index(drop=True)

print(f'Before : {len(df_propositions)} propositions')
print(f'After  : {len(df_clean)} clean propositions')
print(f'Removed: {len(df_propositions) - len(df_clean)} junk rows')

save_path = '/content/drive/My Drive/GP_Project/df_propositions_CLEAN_FINAL.csv'
df_clean.to_csv(save_path, index=False)
print(f'Saved to {save_path}')


In [ ]:
# Run K-Means again on the CLEAN data
from sklearn.cluster import KMeans

num_clusters = 10 # Slightly more clusters to allow better separation
kmeans_clean = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_clean['cluster_id'] = kmeans_clean.fit_predict(clean_vectors)

# Re-calculate the Centroids
from sklearn.metrics import pairwise_distances_argmin_min
closest_indices, _ = pairwise_distances_argmin_min(kmeans_clean.cluster_centers_, clean_vectors)

# Create the NEW Knowledge Map
new_labels = []
for i, idx in enumerate(closest_indices):
    new_labels.append({
        'Cluster_ID': i,
        'Centroid_Summary': df_clean.iloc[idx]['proposition_child']
    })

df_summary_clean = pd.DataFrame(new_labels)
print("✨ NEW Knowledge Map Created on Clean Data!")
print(df_summary_clean)

✨ NEW Knowledge Map Created on Clean Data!
   Cluster_ID                                   Centroid_Summary
0           0  الأرصاد تحذر من حالة الطقس اليوم الخميس 16-4-2...
1           1  إسرائيل تهاجم فرنسا في الأمم المتحدة وسط جدل ع...
2           2  وقف للنار في لبنان رغم الخروقات والجيش الإسرائ...
3           3  تخطى الأكثر قراءة وواصل القراءةالأكثر قراءةلما...
4           4  ورغم تحفظه الجزئي على هذا الطرح، أكد البرغوثي ...
5           5  ​مدن القناة وشمال سيناء: ​الإسماعيلية: الصغرى ...
6           6  وقال جيوفاني ستونوفو المحلل في "يو بي إس" إن "...
7           7  إقرأ المزيد مصادر لـ"رويترز": الهند تسدد ثمن ا...
8           8  وتضمنت الأنشطة، اجتماع رئيس الوزراء مع رئيس مج...
9           9  أم أن الانتقال إلى مرحلة الأبوة هو الذي يؤدي إ...


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans

# 1. Re-initialize the tools (Takes 30 seconds)
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Re-load your saved results from Drive
# (Make sure your drive is mounted!)
df_cluster_summary = pd.read_csv('/content/drive/My Drive/Knowledge_Map_Summary.csv')
# We need to recreate the kmeans object state to use its centers
# We can just manually inject the centers from your summary if needed,
# but usually, you just re-run the Kmeans cell (it's fast) or load the centers:
# For now, let's just define the function to use the centers you already have
cluster_centers = kmeans.cluster_centers_ # If kmeans is still in memory from the clustering cell

def get_single_embedding(text):
    inputs = tokenizer([text], padding=True, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).numpy()

def simulate_rss_ingestion(new_headline):
    from sklearn.metrics.pairwise import cosine_similarity
    new_vector = get_single_embedding(new_headline)
    similarities = cosine_similarity(new_vector, cluster_centers)
    assigned_cluster = np.argmax(similarities)
    confidence_score = similarities[0][assigned_cluster]

    cluster_label = df_cluster_summary.iloc[assigned_cluster]['Centroid_Summary']

    print(f"🚀 [NEW RSS FEED]: {new_headline}")
    print(f"📂 [ROUTED TO CLUSTER]: {assigned_cluster}")
    print(f"🏷️  [TOPIC LABEL]: {cluster_label[:150]}...")
    print(f"✅ [CONFIDENCE]: {confidence_score:.4f}")

# RUN TEST
simulate_rss_ingestion("تحذيرات من عواصف رعدية وأمطار غزيرة على الإسكندرية")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 [NEW RSS FEED]: تحذيرات من عواصف رعدية وأمطار غزيرة على الإسكندرية
📂 [ROUTED TO CLUSTER]: 3
🏷️  [TOPIC LABEL]: إذ رأى أن النظام الجديد لا يقدم فائدة تُذكر لمعظم الناس، بل يُسيء إلى الأنظمة المترية الأخرى، التي اعتبرها، على العكس، مفيدة.مقاطع فيديو قصيرةجهازك لا...
✅ [CONFIDENCE]: 0.6141


## **Phase 3: ChromaDB Ingestion**
> *Goal: Load clean propositions + AraBERT vectors into ChromaDB (HNSW cosine index) with source metadata.*

In [ ]:
!pip install chromadb -q

import chromadb
import numpy as np
import pandas as pd

# Load vectors saved from the embedding step
vectors = np.load('/content/drive/My Drive/proposition_vectors.npy')

# Load the clean propositions (with metadata)
df_clean = pd.read_csv('/content/drive/My Drive/GP_Project/df_propositions_CLEAN_FINAL.csv')

# The clean df was filtered from df_propositions — we need the original row indices
# to pick the matching vectors. Re-apply the same filter on a fresh load.
df_full = pd.read_csv('/content/drive/My Drive/GP_Project/df_propositions_mapped.csv')

noise_keywords = [
    'فيديو', 'تحميل', 'شارك', 'التعليقات', 'جهازك', 'تطبيقات',
    'الرئيسية', 'سجل', 'موقع', 'تابعوا', 'طباعة', 'VK', 'Google'
]
mask = df_full['proposition_child'].str.len() > 25
for word in noise_keywords:
    mask = mask & (~df_full['proposition_child'].str.contains(word, na=False))

clean_indices  = df_full[mask].index.tolist()
clean_vectors  = vectors[clean_indices]

print(f'Clean propositions : {len(df_clean)}')
print(f'Matching vectors   : {clean_vectors.shape}')
assert len(df_clean) == len(clean_vectors), 'Mismatch! Re-run proposition extraction cell first.'
print('Alignment check passed.')


In [ ]:
# Create ChromaDB collection
client = chromadb.Client()

try:
    client.delete_collection('bucket_b_truth')
    print('Old collection deleted.')
except:
    pass

collection = client.create_collection(
    name='bucket_b_truth',
    metadata={'hnsw:space': 'cosine'}
)

# Ingest in batches of 500 (ChromaDB limit per call)
batch_size = 500
total = len(df_clean)

for i in range(0, total, batch_size):
    batch_df  = df_clean.iloc[i:i+batch_size]
    batch_vec = clean_vectors[i:i+batch_size]

    collection.add(
        documents  = batch_df['proposition_child'].tolist(),
        embeddings = batch_vec.tolist(),
        metadatas  = [
            {
                'source':       str(row.get('source', '')),
                'parent_title': str(row.get('parent_title', '')),
                'link':         str(row.get('link', '')),
                'timestamp':    str(row.get('timestamp', '')),
                'bucket':       'B_Truth'
            }
            for _, row in batch_df.iterrows()
        ],
        ids = [str(i + j) for j in range(len(batch_df))]
    )
    print(f'Ingested {min(i+batch_size, total)}/{total}')

print(f'\nDone. Total propositions in ChromaDB: {collection.count()}')


## **Phase 4: Live Fact-Check Demo**
> *Goal: Show that an Arabic claim retrieves specific supporting evidence from Bucket B.*

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = 'aubmindlab/bert-base-arabertv02'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModel.from_pretrained(model_name)
model.eval()

def get_single_embedding(text):
    inputs = tokenizer([text], padding=True, truncation=True,
                       max_length=128, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).numpy()

def fact_check_claim(claim):
    claim_vector = get_single_embedding(claim)
    results = collection.query(
        query_embeddings=claim_vector.tolist(),
        n_results=3
    )
    print(f'CLAIM: {claim}')
    print('=' * 65)
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        confidence = round((1 - dist) * 100, 1)
        print(f'Evidence {i+1} | Source: {meta["source"]} | Match: {confidence}%')
        print(f'  {doc[:220]}')
        print(f'  Link: {meta["link"]}')
        print()


In [ ]:
# Test 1 — Political / Middle East
fact_check_claim('إسرائيل تشن هجوماً على غزة')


In [ ]:
# Test 2 — Economy
fact_check_claim('ارتفاع أسعار الذهب في مصر')


In [ ]:
# Test 3 — Local / Weather
fact_check_claim('تحذيرات من أمطار غزيرة على القاهرة')


In [1]:
import pandas as pd

df_arafacts = pd.read_csv("/Users/russelltamer/Desktop/system 2 RAG/AraFacts 2.csv")
print("Shape:", df_arafacts.shape)
print("Columns:", df_arafacts.columns.tolist())
print()
print(df_arafacts.head(3).to_string())

Shape: (18598, 14)
Columns: ['ClaimID', 'claim', 'description', 'source', 'date', 'source_label', 'normalized_label', 'source_category', 'normalized_category', 'source_url', 'claim_urls', 'evidence_urls', 'evidence_urls.1', 'claim_type']

    ClaimID                                                        claim                                                                                                                            description       source                       date source_label normalized_label source_category normalized_category                                                      source_url                                                                                                                                                                                                                                                                                                                                                                           claim_urls           